# Project 03 (final) — reference solution: spam filter with naive Bayes on real data

**Goal:** you build a complete spam classifier **entirely from scratch** — with
nothing but Bayes' rule, counting and logarithms — and measure it on real data:
5,574 real text messages (about 13 % of them spam) from the *SMS Spam Collection* (UCI).
At the end you compare your model with the scikit-learn implementation.

**This is the fully worked solution.** All cells have been executed.

**Reference to the script:** section 2.4 (Bayes' rule, naive Bayes) and 2.5 (supervised learning).

## 1. Load the data and look at it

Rule number one with real data: **look first, model second**.

**Task:** load `datasets/SMSSpamCollection` (tab-separated, no header row, columns `label` and `text`; `pd.read_csv(..., sep="\t", header=None, names=["label","text"], quoting=3)` — `quoting=3` prevents `"` from being interpreted as a quote character). Then get an overview: class distribution (`value_counts`), spam share (about 13 %), and compare the text length of ham vs. spam (`str.len`, `groupby`).

In [1]:
import os
import pandas as pd

# Also works when the notebook is started from the solution/ folder:
DATA = "datasets/SMSSpamCollection"
if not os.path.exists(DATA):
    DATA = "../" + DATA

df = pd.read_csv(DATA, sep="\t", header=None,
                 names=["label", "text"], quoting=3)  # quoting=3: do not treat " as a quote character
print(df.shape)
df.head()

(5574, 2)


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
print(df["label"].value_counts())
print(f"\nSpam share: {(df['label'] == 'spam').mean():.1%}")

# Are spam messages longer? (Typically: yes, markedly)
df["length"] = df["text"].str.len()
df.groupby("label")["length"].describe().round(1)

label
ham     4827
spam     747
Name: count, dtype: int64

Spam share: 13.4%


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4827.0,71.5,58.3,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.7,28.9,13.0,133.0,149.0,157.0,223.0


**Important — imbalanced classes:** only about 13 % spam. A "classifier" that
stubbornly says *ham* would already have about 87 % accuracy! So accuracy alone
is not an adequate performance measure here — we check this later with precision
and recall.

## 2. Training and test set

We evaluate the model only on messages it has **never seen** during training
(script 2.5: generalisation!). `stratify` ensures that the spam share is the same
in both subsets; the fixed `random_state` makes everything reproducible.

**Task:** split with `train_test_split` into 80 % training / 20 % test (`test_size=0.2`, `random_state=42`, `stratify=labels`) and check that the spam share is about equal in both subsets.

In [3]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

print(f"Training: {len(train_texts)} messages, test: {len(test_texts)} messages")
print(f"Spam share training: {(train_labels == 'spam').mean():.1%}, "
      f"test: {(test_labels == 'spam').mean():.1%}")

Training: 4459 messages, test: 1115 messages
Spam share training: 13.4%, test: 13.4%


## 3. From text to words: tokenisation

Naive Bayes works with word probabilities — so we have to break messages into
words. We keep it deliberately simple: lowercase, then extract all sequences of
letters and digits.

**Task:** implement `tokenize(text)` yourself.

**Hint:** use the regex `[a-z0-9']+` and lowercase the text beforehand (`.lower()`). **Self-check:** `tokenize("WINNER!! Claim your £900 prize now!")` must give `['winner','claim','your','900','prize','now']`.

In [4]:
import re

WORD_PATTERN = re.compile(r"[a-z0-9']+")

def tokenize(text):
    """Splits a text into a list of lowercased words."""
    return WORD_PATTERN.findall(text.lower())

# Mini test — must print True:
print(tokenize("WINNER!! Claim your £900 prize now!") == ["winner", "claim", "your", "900", "prize", "now"])

True


## 4. Naive Bayes from scratch

As a reminder (script 2.4): for a message with words $w_1, \dots, w_n$ we compare

$$P(\text{spam} \mid w_1..w_n) \propto P(\text{spam}) \prod_i P(w_i \mid \text{spam})
\qquad \text{vs.} \qquad
P(\text{ham} \mid w_1..w_n) \propto P(\text{ham}) \prod_i P(w_i \mid \text{ham})$$

Two practical tricks, both of which you implement yourself:

1. **Log probabilities:** the product of hundreds of small numbers would collapse
   numerically to 0 (*underflow*). We therefore compute with sums of logarithms:
   $\log P(c) + \sum_i \log P(w_i \mid c)$ — the comparison stays the same,
   because the logarithm is monotonic.
2. **Laplace smoothing:** a word that never occurred in spam during training would
   have $P(w \mid \text{spam}) = 0$ — one single such word would "acquit" every
   spam message ($\log 0 = -\infty$). We therefore pretend we had seen every known
   word in every class **one extra time** ($\alpha = 1$):

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{total words in } c + |V|}$$

where $|V|$ is the size of the vocabulary (all known words).

**Task:** build `train(texts, labels)` (counts the words per class, the vocabulary and the priors), `log_word_prob(model, word, cls)` (the Laplace formula above), `log_posterior(model, text, cls)` (prior + sum of the log word probabilities over the known words) and `classify(model, text)`. **Self-check:** `"URGENT! You have won a free prize, call now!"` → `spam`, `"Ok, see you at the station at 6"` → `ham`.

In [5]:
from collections import Counter
import math

def train(texts, labels):
    """Counts everything naive Bayes needs. Returns a model dictionary."""
    n_messages = Counter(labels)                        # {"ham": ..., "spam": ...}
    word_counts = {"ham": Counter(), "spam": Counter()} # word -> count per class
    for text, label in zip(texts, labels):
        word_counts[label].update(tokenize(text))
    vocabulary = set(word_counts["ham"]) | set(word_counts["spam"])
    return {
        "log_prior": {c: math.log(n_messages[c] / len(labels)) for c in ("ham", "spam")},
        "word_counts": word_counts,
        "total": {c: sum(word_counts[c].values()) for c in ("ham", "spam")},
        "V": len(vocabulary),
        "vocabulary": vocabulary,
    }

def log_word_prob(model, word, cls):
    """log P(word | cls) with Laplace smoothing (alpha = 1)."""
    numerator = model["word_counts"][cls][word] + 1
    denominator = model["total"][cls] + model["V"]
    return math.log(numerator / denominator)

def log_posterior(model, text, cls):
    """log( P(cls) * prod P(w|cls) ) — unknown words are ignored."""
    result = model["log_prior"][cls]
    for word in tokenize(text):
        if word in model["vocabulary"]:
            result += log_word_prob(model, word, cls)
    return result

def classify(model, text):
    return "spam" if log_posterior(model, text, "spam") > log_posterior(model, text, "ham") else "ham"

model = train(train_texts, train_labels)
print(f"Vocabulary: {model['V']} words")
print(classify(model, "URGENT! You have won a free prize, call now!"))   # expected: spam
print(classify(model, "Ok, see you at the station at 6"))                # expected: ham

Vocabulary: 7899 words
spam
ham


## 5. How good is the filter really?

Now the acid test on the held-out test data. Besides accuracy we look at the
**confusion matrix** and two figures:

- **Precision** (spam): when the filter says "spam" — how often is that right?
  *(Important: false positives = genuine messages in the spam folder = very annoying!)*
- **Recall** (spam): how much of the actual spam does the filter catch?

**Task:** compute `accuracy_score`, `confusion_matrix` and `classification_report` on the test data. **Expectation:** accuracy about 0.98 — clearly above the "always ham" baseline (about 0.87). Pay particular attention to precision and recall of the spam class.

In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

predictions = [classify(model, t) for t in test_texts]

print(f"Accuracy: {accuracy_score(test_labels, predictions):.4f}")
print(f"(For comparison — always saying 'ham': {(test_labels == 'ham').mean():.4f})\n")
print(pd.DataFrame(confusion_matrix(test_labels, predictions, labels=["ham", "spam"]),
                   index=["actual: ham", "actual: spam"], columns=["predicted: ham", "predicted: spam"]))
print()
print(classification_report(test_labels, predictions, digits=3))

Accuracy: 0.9857
(For comparison — always saying 'ham': 0.8664)

              predicted: ham  predicted: spam
actual: ham              964                2
actual: spam              14              135

              precision    recall  f1-score   support

         ham      0.986     0.998     0.992       966
        spam      0.985     0.906     0.944       149

    accuracy                          0.986      1115
   macro avg      0.986     0.952     0.968      1115
weighted avg      0.986     0.986     0.985      1115



## 6. What did the model learn?

A big advantage of naive Bayes: you can **look inside**. Which words argue most
strongly for spam? We rank by the log ratio
$\log \frac{P(w \mid \text{spam})}{P(w \mid \text{ham})}$ (only words occurring at least 5 times).

**Hint:** rank the words by the log ratio $\log P(w\mid\text{spam}) - \log P(w\mid\text{ham})$ and consider only words that occur at least 5 times in total.

In [7]:
def spam_evidence(word):
    return log_word_prob(model, word, "spam") - log_word_prob(model, word, "ham")

frequent = [w for w in model["vocabulary"]
            if model["word_counts"]["spam"][w] + model["word_counts"]["ham"][w] >= 5]
top_spam = sorted(frequent, key=spam_evidence, reverse=True)[:15]
top_ham = sorted(frequent, key=spam_evidence)[:15]
print("Strongest spam words:", ", ".join(top_spam))
print("\nStrongest ham words: ", ", ".join(top_ham))

Strongest spam words: claim, prize, won, 150p, tone, 18, guaranteed, cs, 500, awarded, 1000, landline, 150ppm, uk, www

Strongest ham words:  gt, lt, he, i'll, da, lor, later, she, amp, ask, anything, doing, cos, home, said


## 7. Comparison with scikit-learn

To finish, the same model with the standard tools of practice:
`CountVectorizer` (counts words) + `MultinomialNB` (exactly our algorithm).
If your by-hand version is good, the two are close together.

**Hint:** `CountVectorizer(token_pattern=r"[a-z0-9']+", lowercase=True)` + `MultinomialNB(alpha=1.0)`. Your by-hand accuracy should be very close to the scikit-learn accuracy.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = CountVectorizer(token_pattern=r"[a-z0-9']+", lowercase=True)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

sk_model = MultinomialNB(alpha=1.0)
sk_model.fit(X_train, train_labels)
sk_accuracy = sk_model.score(X_test, test_labels)

my_accuracy = accuracy_score(test_labels, predictions)
print(f"My naive Bayes:           {my_accuracy:.4f}")
print(f"scikit-learn multinomial: {sk_accuracy:.4f}")

My naive Bayes:           0.9857
scikit-learn multinomial: 0.9857


*(Small deviations are normal: scikit-learn counts words that occur several times
in a message with their multiplicity, while details such as the handling of unknown
words are solved slightly differently in our version.)*

## Done — what you can do now

- load a real data set, explore it and split it cleanly into train/test
- translate Bayes' rule into a working classifier
  (including the two practical tricks: log space and Laplace smoothing)
- evaluate a model with the *right* metrics (precision/recall instead of accuracy alone)
- benchmark your by-hand model against an industry implementation

**Bonus tasks** (optional, no reference solution):
1. The filter puts some genuine messages into the spam folder (false positives). Look
   at those messages (`test_texts[(predictions_series == "spam") & (test_labels == "ham")]`) — why does the model stumble?
2. Experiment with the smoothing: what happens at $\alpha = 0.01$ or $\alpha = 10$?
3. Remove words that occur only once from the vocabulary. Does the model get better or
   worse — and why could either happen?